<a href="https://colab.research.google.com/github/ctrlcoded/Resolving-Idioms-to-Improve-the-Machine-Translation-in-Indic-Languages/blob/main/m2m_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers sentencepiece sacrebleu evaluate tqdm pandas
!pip install rouge_score # Often required as a dependency for some evaluators

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=00fb3b20e1c8e4adab129130f3872e5c6aa8cd2ab2fbaa244d97587e9887aeda
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
import pandas as pd
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch
from tqdm import tqdm

# Load your CSV
df = pd.read_csv('data.csv')

# Initialize Model and Tokenizer
model_name = "facebook/m2m100_418M" # You can use "facebook/m2m100_1.2B" for better accuracy if using a GPU
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Set language codes
tokenizer.src_lang = "hi"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

In [3]:
def translate_hindi_to_english(text):
    encoded_hi = tokenizer(text, return_tensors="pt").to(device)
    generated_tokens = model.generate(
        **encoded_hi,
        forced_bos_token_id=tokenizer.get_lang_id("en")
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# Apply translation
tqdm.pandas()
df['machine_translation'] = df['hindi'].progress_apply(translate_hindi_to_english)

100%|██████████| 100/100 [16:12<00:00,  9.73s/it]


In [4]:
import evaluate

bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

# Prepare data for metrics
predictions = df['machine_translation'].tolist()
# BLEU expects a list of lists for references
references = [[rel] for rel in df['reference_translation'].tolist()]

# Calculate BLEU
bleu_results = bleu.compute(predictions=predictions, references=references)

# Calculate METEOR
# Meteor expects a flat list for both
meteor_results = meteor.compute(predictions=predictions, references=df['reference_translation'].tolist())

print(f"BLEU Score: {bleu_results['score']:.2f}")
print(f"METEOR Score: {meteor_results['meteor']:.2f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


BLEU Score: 13.34
METEOR Score: 0.43


In [5]:
df.to_csv('translated_idioms_results.csv', index=False)